<a href="https://colab.research.google.com/github/mariuszpanczyk/meningococcal-imc-analysis/blob/main/kawasaki_complete_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kawasaki disease: complete statistical analysis

This notebook reproduces the complete statistical workflow for the study. The source dataset is not distributed with this repository because it contains participant-level clinical data. Authorized users may execute the notebook after placing the source workbook in the private Google Colab session when prompted. The notebook exports aggregate statistical results only.

In [ ]:
# 0. Install/import packages
!pip -q install openpyxl statsmodels scikit-learn
import sys, json, hashlib, zipfile, unicodedata, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import StratifiedTable
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
SEED=20261018; N_PERM=19999; N_BOOT=5000; ALPHA=.05
rng=np.random.default_rng(SEED)
OUT=Path('/content/Kawasaki_global_results'); OUT.mkdir(exist_ok=True)
print(sys.version); print('pandas',pd.__version__,'numpy',np.__version__)

In [ ]:
# 1. Upload the private source workbook
from google.colab import files
uploaded=files.upload(); names=[n for n in uploaded if n.lower().endswith(('.xlsx','.csv'))]
if not names: raise FileNotFoundError('Upload the source .xlsx or .csv database.')
name=names[0]; path=Path('/content')/name
if path.suffix.lower()=='.csv':
    df=pd.read_csv(path,sep=None,engine='python')
else:
    xls=pd.ExcelFile(path,engine='openpyxl'); print('Sheets:',xls.sheet_names)
    sheet='FINAL KD5 WERSJA DO OBLICZEŃ' if 'FINAL KD5 WERSJA DO OBLICZEŃ' in xls.sheet_names else xls.sheet_names[0]
    df=pd.read_excel(path,sheet_name=sheet,engine='openpyxl')
sha=hashlib.sha256(path.read_bytes()).hexdigest(); df.columns=[str(c).strip() for c in df.columns]; df=df.loc[df['Nr ID'].notna()].copy()
print(name,df.shape,'SHA-256',sha)

In [ ]:
# 2. Harmonisation, validation, derived variables
def yesno(x):
    if pd.isna(x): return np.nan
    s=str(x).strip().lower().replace(',','.')
    if s in {'1','1.0','yes','y','tak','true'}: return 1.0
    if s in {'2','2.0','0','0.0','no','n','nie','false'}: return 0.0
    return np.nan
ID='Nr ID'; df[ID]=pd.to_numeric(df[ID],errors='coerce').astype('Int64')
if df[ID].duplicated().any(): raise ValueError('Duplicated IDs detected.')
df['Group2']=df['Group'].astype(str).str.strip().str.upper().map({'KD':'KD','CONTROL':'HC'})
df['Female']=df['Sex'].astype(str).str.strip().str.upper().map({'K':1,'F':1,'M':0})
BINARY=['Chol Total Borderline','Chol Total High','HDL Borderline','HDL Low','non-HDL Borderline','non-HDL High','Trig Borderline','Trig High','IFG','Glucose high','cIMT >95 pc','PWV >95 pc','AoSBP >95 c','KD_anemia','KD_EF <55%','KD_aneurysms','KD_aneurysms_persistnet','KD_IVIG resistance']
for c in BINARY:
    if c in df: df[c+'_bin']=df[c].map(yesno)
num_exclude={ID,'Data badania','Data urodzenia'}
for c in df.columns:
    if c not in num_exclude and not c.endswith('_bin') and c not in ['Group','Group2','Sex']:
        df[c]=pd.to_numeric(df[c],errors='ignore')
print(df.Group2.value_counts(dropna=False)); display(df[['Nr ID','Group2','Sex']].head())

## Analysis A. Descriptive characteristics and KD vs healthy-control comparisons
Continuous variables are reported by group. Between-group inference uses the Brunner–Munzel statistic and a label-permutation p value. The probabilistic index P(KD > HC), equivalent to the Mann–Whitney relative effect, is accompanied by a stratified percentile-bootstrap 95% CI.

In [ ]:
# 3. Continuous-variable engine
TABLE1=['Age','BMI','Hgb','Chol Total','HDL','non-HDL','Trig','Glucose','sVCAM','Galectin-3','CEC (kom/ml)']
TABLE2=['SBP','DBP','SBP Z-score','DBP Z-score','cIMT','cIMT Z-score','PWV','PWV Z-score','AoSBP','AoSBP Z-score','AoDBP','AoMAP','AoPP','Ap@HR75']
def prob_index(x,y):
    x=np.asarray(x); y=np.asarray(y); return (np.sum(x[:,None]>y)+.5*np.sum(x[:,None]==y))/(len(x)*len(y))
def continuous_compare(var):
    a=pd.to_numeric(df.loc[df.Group2.eq('KD'),var],errors='coerce').dropna().to_numpy()
    b=pd.to_numeric(df.loc[df.Group2.eq('HC'),var],errors='coerce').dropna().to_numpy()
    bm=stats.brunnermunzel(a,b,alternative='two-sided',distribution='t')
    obs=prob_index(a,b); pool=np.r_[a,b]; n=len(a)
    perm=np.empty(N_PERM)
    rr=np.random.default_rng(SEED+sum(map(ord,var)))
    for i in range(N_PERM):
        z=rr.permutation(pool); perm[i]=prob_index(z[:n],z[n:])
    pperm=(1+np.sum(np.abs(perm-.5)>=abs(obs-.5)))/(N_PERM+1)
    boot=np.empty(N_BOOT)
    for i in range(N_BOOT): boot[i]=prob_index(rr.choice(a,len(a),True),rr.choice(b,len(b),True))
    return {'variable':var,'KD_n':len(a),'KD_mean':a.mean(),'KD_SD':a.std(ddof=1),'KD_median':np.median(a),'KD_IQR':np.subtract(*np.percentile(a,[75,25])),'HC_n':len(b),'HC_mean':b.mean(),'HC_SD':b.std(ddof=1),'HC_median':np.median(b),'HC_IQR':np.subtract(*np.percentile(b,[75,25])),'relative_effect':obs,'CI95_low':np.quantile(boot,.025),'CI95_high':np.quantile(boot,.975),'BM_statistic':bm.statistic,'BM_p':bm.pvalue,'permutation_p':pperm}
table1_cont=pd.DataFrame([continuous_compare(v) for v in TABLE1 if v in df])
table2_cont=pd.DataFrame([continuous_compare(v) for v in TABLE2 if v in df])
display(table1_cont.round(3)); display(table2_cont.round(3))

In [ ]:
# 4. Binary categorical comparisons: Fisher exact test and exact/Woolf OR CI
CAT=[c for c in BINARY if c in df]
def cat_compare(c):
    v=c+'_bin'; d=df.loc[df.Group2.isin(['KD','HC']),['Group2',v]].dropna()
    tab=pd.crosstab(d.Group2,d[v]).reindex(index=['KD','HC'],columns=[1.,0.],fill_value=0)
    a,b=tab.loc['KD',1.],tab.loc['KD',0.]; cc,dd=tab.loc['HC',1.],tab.loc['HC',0.]
    OR,p=stats.fisher_exact([[a,b],[cc,dd]])
    aa,bb,ccc,ddd=[x+.5 if min(a,b,cc,dd)==0 else x for x in (a,b,cc,dd)]
    se=np.sqrt(1/aa+1/bb+1/ccc+1/ddd); lor=np.log((aa*ddd)/(bb*ccc)); lo,hi=np.exp([lor-1.96*se,lor+1.96*se])
    return {'variable':c,'KD_yes':a,'KD_no':b,'HC_yes':cc,'HC_no':dd,'OR_KD_vs_HC':OR,'CI95_low':lo,'CI95_high':hi,'Fisher_p':p}
categorical=pd.DataFrame([cat_compare(c) for c in CAT]); display(categorical.round(4))

## Analysis B. Sex-stratified and group-by-sex analyses

In [ ]:
# 5. Cochran–Mantel–Haenszel analyses for binary variables, stratified by sex
cmh=[]
for c in CAT:
    tabs=[]
    for sx in [0,1]:
        d=df.loc[df.Female.eq(sx),['Group2',c+'_bin']].dropna()
        t=pd.crosstab(d.Group2,d[c+'_bin']).reindex(index=['KD','HC'],columns=[1.,0.],fill_value=0).to_numpy()
        if t.sum()>0: tabs.append(t)
    if len(tabs)==2:
        st=StratifiedTable(tabs)
        cmh.append({'variable':c,'pooled_OR':st.oddsratio_pooled,'CMH_p':st.test_null_odds().pvalue,'Breslow_Day_p':st.test_equal_odds(adjust=True).pvalue})
cmh_results=pd.DataFrame(cmh); display(cmh_results.round(4))

In [ ]:
# 6. OLS-HC3 group, sex, and group×sex models for continuous variables
# Robust HC3 inference is used as a reproducible Python analogue for factorial robustness.
twoway=[]
for y in list(dict.fromkeys(TABLE1+TABLE2)):
    if y not in df: continue
    d=df[[y,'Group2','Female']].dropna().copy()
    if len(d)<8: continue
    fit=smf.ols(f'Q("{y}") ~ C(Group2) * C(Female)',data=d).fit(cov_type='HC3')
    for term in fit.params.index:
        if term!='Intercept':
            ci=fit.conf_int().loc[term]
            twoway.append({'outcome':y,'term':term,'n':len(d),'beta':fit.params[term],'HC3_SE':fit.bse[term],'CI95_low':ci[0],'CI95_high':ci[1],'p':fit.pvalues[term]})
twoway_results=pd.DataFrame(twoway); display(twoway_results.round(4))

## Analysis C. Acute-phase severity markers and follow-up outcomes in KD

In [ ]:
# 7. Pairwise-complete Spearman correlations in KD
kd=df.loc[df.Group2.eq('KD')].copy()
PRED=['KD_max CRP','KD_max PCT','KD_min lymph','KD_min Hgb','KD_min albumin','KD_min EF ','KD_anuerysms Z-score']
OUTCOMES=[c for c in ['WBC','Lymph','Hgb','Chol Total','HDL','non-HDL','Trig','Glucose','SBP','SBP Z-score','DBP','DBP Z-score','cIMT','cIMT Z-score','PWV','PWV Z-score','AoSBP','AoSBP Z-score','AoDBP','AoMAP','AoPP','Ap@HR75','Galectin-3','sVCAM','LVIDd','IVSD','PWLVD','LV mass','LVMI [g/m2]','LVMI [g/m2,7]','EF (Simpson)','LVM Z-score for height','LMM Z-score for LBM','CEC (kom/ml)'] if c in kd]
corr=[]
for x in PRED:
    if x not in kd: continue
    for y in OUTCOMES:
        d=kd[[x,y]].apply(pd.to_numeric,errors='coerce').dropna()
        if len(d)>=4:
            r,pv=stats.spearmanr(d[x],d[y]); corr.append({'predictor':x,'outcome':y,'n':len(d),'rho':r,'p':pv})
correlations=pd.DataFrame(corr); correlations['p_BH']=multipletests(correlations.p,method='fdr_bh')[1]
display(correlations.sort_values('p').head(25).round(4))

## Analysis D. Prespecified outcomes by CAA and IVIG resistance
The prespecified outcomes are analysed in the Kawasaki disease subgroup using available cases.

In [ ]:
# 8. CAA and IVIG outcome comparisons: Yuen trimmed-mean test + stratified bootstrap
CLIN_OUT=[c for c in ['Glucose','SBP Z-score','DBP Z-score','cIMT Z-score','PWV Z-score','AoSBP Z-score','Galectin-3','CEC (kom/ml)'] if c in kd]
def yuen_boot(var,grp,trim=.2):
    d=kd[[var,grp]].dropna(); a=pd.to_numeric(d.loc[d[grp].eq(1),var],errors='coerce').dropna().to_numpy(); b=pd.to_numeric(d.loc[d[grp].eq(0),var],errors='coerce').dropna().to_numpy()
    if min(len(a),len(b))<3: return None
    test=stats.ttest_ind(a,b,equal_var=False,trim=trim)
    rr=np.random.default_rng(SEED+sum(map(ord,var+grp))); obs=stats.trim_mean(a,trim)-stats.trim_mean(b,trim); bs=[]
    for _ in range(N_BOOT): bs.append(stats.trim_mean(rr.choice(a,len(a),True),trim)-stats.trim_mean(rr.choice(b,len(b),True),trim))
    bs=np.asarray(bs); pboot=2*min(np.mean(bs<=0),np.mean(bs>=0))
    return {'grouping':grp,'outcome':var,'n_yes':len(a),'n_no':len(b),'trimmed_mean_difference':obs,'Yuen_t':test.statistic,'Yuen_p':test.pvalue,'bootstrap_CI95_low':np.quantile(bs,.025),'bootstrap_CI95_high':np.quantile(bs,.975),'bootstrap_p':pboot}
comparisons=[]
for g in ['KD_aneurysms_bin','KD_IVIG resistance_bin']:
    if g in kd:
        for y in CLIN_OUT:
            r=yuen_boot(y,g)
            if r: comparisons.append(r)
clinical_comparisons=pd.DataFrame(comparisons); clinical_comparisons['p_BH']=clinical_comparisons.groupby('grouping').Yuen_p.transform(lambda p:multipletests(p,method='fdr_bh')[1])
display(clinical_comparisons.round(4))

In [ ]:
# 9. IVIG resistance regression: primary OLS-HC3 and age/sex/BMI-z adjusted models
reg=[]; fitted={}
for y in CLIN_OUT:
    for adj in [False,True]:
        cols=[y,'KD_IVIG resistance_bin']+(['Age','Female','BMI Z-score'] if adj else [])
        d=kd[cols].dropna()
        form=f'Q("{y}") ~ Q("KD_IVIG resistance_bin")'+(' + Age + Female + Q("BMI Z-score")' if adj else '')
        fit=smf.ols(form,data=d).fit(cov_type='HC3'); term='Q("KD_IVIG resistance_bin")'; ci=fit.conf_int().loc[term]
        reg.append({'outcome':y,'model':'adjusted' if adj else 'primary','n':len(d),'beta':fit.params[term],'HC3_SE':fit.bse[term],'CI95_low':ci[0],'CI95_high':ci[1],'p':fit.pvalues[term],'R2':fit.rsquared})
        fitted[(y,adj)]=(fit,d,form)
regression=pd.DataFrame(reg); regression['p_BH']=regression.groupby('model').p.transform(lambda p:multipletests(p,method='fdr_bh')[1]); display(regression.round(4))

In [ ]:
# 10. Diagnostics, leave-one-out, participant bootstrap for primary IVIG models
diagn=[]; loo=[]; boot=[]
for y in CLIN_OUT:
    fit,d,form=fitted[(y,False)]; classical=smf.ols(form,data=d).fit(); inf=classical.get_influence(); cook=inf.cooks_distance[0]
    bp=sm.stats.diagnostic.het_breuschpagan(classical.resid,classical.model.exog)
    diagn.append({'outcome':y,'n':len(d),'Shapiro_p':stats.shapiro(classical.resid).pvalue,'Breusch_Pagan_p':bp[1],'max_Cooks_D':max(cook),'n_Cooks_over_4n':sum(cook>4/len(d))})
    for ix in d.index:
        f=smf.ols(form,data=kd.drop(index=ix)).fit(cov_type='HC3'); term='Q("KD_IVIG resistance_bin")'
        loo.append({'outcome':y,'omitted_ID':int(kd.loc[ix,ID]),'beta_LOO':f.params[term],'p_LOO':f.pvalues[term]})
    rr=np.random.default_rng(SEED+sum(map(ord,y))); vals=[]
    for _ in range(N_BOOT):
        s=d.iloc[rr.integers(0,len(d),len(d))]
        if s['KD_IVIG resistance_bin'].nunique()==2: vals.append(smf.ols(form,data=s).fit().params['Q("KD_IVIG resistance_bin")'])
    vals=np.asarray(vals); b=fit.params['Q("KD_IVIG resistance_bin")']
    boot.append({'outcome':y,'beta':b,'CI95_low':np.quantile(vals,.025),'CI95_high':np.quantile(vals,.975),'successful':len(vals),'sign_stability_pct':100*np.mean(np.sign(vals)==np.sign(b))})
diagnostics=pd.DataFrame(diagn); leave_one_out=pd.DataFrame(loo); bootstrap=pd.DataFrame(boot)
display(diagnostics.round(4)); display(bootstrap.round(4))

## Analysis E. Exploratory ROC analyses for CAA and IVIG resistance

In [ ]:
# 11. ROC curves, bootstrap AUC CI, Youden threshold, diagnostic accuracy
def roc_analysis(outcome,predictor):
    d=kd[[outcome,predictor]].dropna(); yy=d[outcome].astype(int).to_numpy(); score=pd.to_numeric(d[predictor],errors='coerce').to_numpy()
    if len(np.unique(yy))<2:return None,None
    auc=roc_auc_score(yy,score); direction=1
    if auc<.5: score=-score; auc=roc_auc_score(yy,score); direction=-1
    fpr,tpr,thr=roc_curve(yy,score); j=np.argmax(tpr-fpr); cut=thr[j]
    pred=(score>=cut).astype(int); tn,fp,fn,tp=confusion_matrix(yy,pred).ravel(); sens=tp/(tp+fn); spec=tn/(tn+fp); ppv=tp/(tp+fp) if tp+fp else np.nan; npv=tn/(tn+fn) if tn+fn else np.nan
    rr=np.random.default_rng(SEED+sum(map(ord,predictor+outcome))); ba=[]
    for _ in range(N_BOOT):
        ix=rr.integers(0,len(yy),len(yy));
        if len(np.unique(yy[ix]))==2: ba.append(roc_auc_score(yy[ix],score[ix]))
    row={'outcome':outcome,'predictor':predictor,'n':len(yy),'AUC':auc,'AUC_CI95_low':np.quantile(ba,.025),'AUC_CI95_high':np.quantile(ba,.975),'direction':direction,'Youden_cutoff_original_scale':cut*direction,'sensitivity':sens,'specificity':spec,'PPV':ppv,'NPV':npv,'accuracy':(tp+tn)/len(yy),'LR_positive':sens/(1-spec) if spec<1 else np.inf,'LR_negative':(1-sens)/spec if spec>0 else np.inf}
    return row,(fpr,tpr)
roc_rows=[]; curves={}
for outcome in ['KD_IVIG resistance_bin','KD_aneurysms_bin']:
    if outcome not in kd: continue
    for y in CLIN_OUT:
        r,c=roc_analysis(outcome,y)
        if r: roc_rows.append(r); curves[(outcome,y)]=c
roc_results=pd.DataFrame(roc_rows); display(roc_results.round(4))
for outcome in roc_results.outcome.unique():
    fig,ax=plt.subplots(figsize=(6,6))
    for _,r in roc_results.loc[roc_results.outcome.eq(outcome)].iterrows():
        fpr,tpr=curves[(outcome,r.predictor)]; ax.plot(fpr,tpr,label=f'{r.predictor} (AUC={r.AUC:.2f})')
    ax.plot([0,1],[0,1],'k--'); ax.set(xlabel='1 − specificity',ylabel='Sensitivity',title=outcome.replace('_bin','')); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(OUT/f'ROC_{outcome}.png',dpi=300); plt.show()

In [ ]:
# 12. Export all results, metadata, and figures
objects={'Table1_continuous':table1_cont,'Table2_continuous':table2_cont,'Categorical':categorical,'CMH_sex_stratified':cmh_results,'Group_sex_HC3':twoway_results,'Spearman_KD':correlations,'CAA_IVIG_comparisons':clinical_comparisons,'IVIG_regression_HC3':regression,'IVIG_diagnostics':diagnostics,'IVIG_leave_one_out':leave_one_out,'IVIG_bootstrap':bootstrap,'ROC':roc_results}
for k,v in objects.items(): v.to_csv(OUT/f'{k}.csv',index=False)
meta={'source_file':name,'sha256':sha,'seed':SEED,'permutations':N_PERM,'bootstrap_replicates':N_BOOT,'available_case_analysis':True,'python':sys.version,'pandas':pd.__version__}
with open(OUT/'metadata.json','w') as f: json.dump(meta,f,indent=2)
with pd.ExcelWriter(OUT/'Kawasaki_complete_results.xlsx',engine='openpyxl') as w:
    pd.DataFrame([meta]).to_excel(w,'Metadata',index=False)
    for k,v in objects.items(): v.to_excel(w,k[:31],index=False)
zip_path=Path('/content/Kawasaki_global_results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir(): z.write(f,arcname=f.name)
print('Complete:',zip_path)
files.download(str(zip_path))